# Анализ монетизации онлайн-игры

Заказчик — онлайн-игра «Секреты Темнолесья», в которой каждый игрок следует по сюжету своего персонажа, выполняет задания и исследует огромный виртуальный мир. Продажа внутриигровой валюты «райские лепестки» — это основная часть дохода команды разработки. Команда игры планирует привлекать платящих игроков и продвигать покупку эпических предметов с помощью рекламы. Необходимо решить несколько задач: 
- выяснить, какая доля игроков покупает внутриигровую валюту «райские лепестки» за реальные деньги и есть ли зависимость доли платящих игроков от расы персонажа;
- детально изучить, как происходит покупка эпических предметов внутри игры.

Данные об активности игроков «Секретов Темнолесья» хранятся в схеме `fantasy` и содержат информацию о зарегистрированных игроках и их персонажах, а также сведения о совершённых внутриигровых покупках за игровую валюту «райские лепестки».

## Что нужно сделать
Задачи: провести разведочный и исследовательский анализ данных, проверить наличие зависимости активности игроков от расы персонажа, подготовить общие выводы и рекомендации.

## 1. Разведочный анализ данных

### 1.1. Информация о таблицах

Выведем названия всех таблиц схемы fantasy.

In [ ]:
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'fantasy'

Схема `fantasy` содержит семь таблиц: `classes`, `country`, `users`, `events`, `items`, `skills`, `race`.  
К ключевым таблицам можно отнести users с информацией об игроках и events с информацией о внутриигровых покупках с использованием игровой валюты «райские лепестки». Остальные таблицы содержат более детальную информацию об игроках и эпических предметах. 

### 1.2. Данные в таблице users

Начнем изучение данных с таблицы users. Получим информацию о названии полей таблицы и типе данных в них, а также информацию о первичных и внешних ключах. Итоговая таблица должна содержать такие поля: 
-    table_schema — название схемы;
-    table_name — название таблицы;
-    column_name — название поля;
-    data_type — тип данных, которые хранятся в поле;
-    constraint_name — информация о первичном или внешнем ключе. Поле может содержать значения NULL — это значит, что поле не имеет ограничений и не используется в качестве первичного или внешнего ключа.

In [ ]:
SELECT c.table_schema, c.table_name, c.column_name, c.data_type, constraint_name
FROM information_schema.columns as c 
LEFT JOIN information_schema.key_column_usage AS k
    ON c.table_schema = k.table_schema 
    AND c.table_name = k.table_name
    AND c.column_name = k.column_name
WHERE c.table_schema = 'fantasy' and c.table_name = 'users'

table_schema |	table_name |	column_name |	data_type |	constraint_name
:-------: | :-------: | :-------: | :-------: | -------:
fantasy |	users |	id |	character varying |	users_pkey
fantasy |	users |	tech_nickname |	character varying |	
fantasy |	users |	class_id |	character varying |	users_class_id_fkey
fantasy |	users |	ch_id |	character varying |	users_ch_id_fkey
fantasy |	users |	birthdate |	character varying |	
fantasy |	users |	pers_gender |	character varying |	
fantasy |	users |	registration_dt |	character varying |	
fantasy |	users |	server |	character varying |	
fantasy |	users |	race_id |	character varying |	users_race_id_fkey
fantasy |	users |	payer |	integer	|
fantasy |	users |	loc_id |	character varying |	users_loc_id_fkey

Таблица `users` содержит 11 полей, и большинство из них хранят текстовые данные. При этом поле `id` с идентификатором игрока — это первичный ключ таблицы, а четыре поля `class_id`, `ch_id`, `race_id` и `loc_id` — внешние ключи. Можно предположить, что таблица `users` связана с таблицами `classes`, `skills`, `race` и `country`.

### 1.3. Вывод первых строк таблицы users

Теперь познакомимся с данными — выведем первые пять строк таблицы `users`. При этом в выдачу добавим поле row_count с подсчётом общего количества строк в таблице.

In [ ]:
SELECT *, count(*) over() as row_count
FROM fantasy.users
LIMIT 5

id | tech_nickname | class_id | ch_id |	birthdate |	pers_gender | registration_dt |	server | race_id | payer | loc_id | row_count
:-------: | :------- | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | -------:
00-0037846|	DivineBarbarian4154|	9RD|JJR2|	6/4/1994|	Male|	1/20/2005|	server_1|	B1|	0|	US|	22214
00-0041533|	BoldInvoker7693|	Z3Q|	HQ9N|	6/29/1987|	Male|	4/8/2022|	server_1|	R2|	0|	US|	22214
00-0045747|	NobleAlchemist7633|	382|	IXBW|	7/29/1992|	Male|	10/12/2013|	server_1|	K3|	0|	US|	22214
00-0055274|	SteadfastArcher8318	|ZD0|	QSUB|	9/14/1985|	Female|	4/10/2008|	server_1|	R2|	0|	US|	22214
00-0076100|	RadiantProphet353|	YC8|	HQ9N|	4/11/1997|	Female|	9/29/2013|	server_2|	K4|	1|	US|	22214

Теперь можно зафиксировать содержимое строк и отметить возможные сложности, например, формат представления даты. Всего данные содержат информацию о 22214 игроках.

### 1.4. Проверка пропусков в таблице users

В данных присутствует первичный ключ — идентификатор игрока, поэтому проблему с полными дубликатами строк можно исключить. Проверим другой тип ошибки — пропуски.  
Посчитаем общее количество строк с пропусками в любом из полей, которые понадобятся при анализе: `class_id`, `ch_id`, `pers_gender`, `server`, `race_id`, `payer`, `loc_id`.

In [ ]:
SELECT count(id)
FROM fantasy.users
WHERE class_id IS NULL OR ch_id IS NULL OR pers_gender IS NULL OR
    server IS NULL OR race_id IS NULL OR payer IS NULL OR loc_id IS NULL

Хорошие новости — пропусков в данных нет!

### 1.5. Знакомство с категориальными данными таблицы users

Таблица `users` содержит информацию об игроках. В данных можно выделить такие категории: характеристика персонажа, его пол, класс, легендарный навык, а также страна регистрации игрока или игровой сервер.
Выведем уникальные значения в поле `server` таблицы `users` и для каждого сервера найдем количество строк.

In [ ]:
select distinct server, count(id)
from fantasy.users
group by (server)

server|	count
:------- | -------:
server_1|	16715
server_2|	5499

Игрокам доступно два сервера. При этом на первом сервере примерно в три раза больше игроков, чем на втором.

### 1.6. Знакомство с таблицей events

Вторая таблица, с которой нужно поработать, — это таблица с информацией о внутриигровых покупках `events`. Изучим её более внимательно, как это делали с таблицей `users`.
    Выведем названия всех полей, их тип данных и информацию о ключевых полях таблицы `events`.

In [ ]:
SELECT c.table_schema,
       c.table_name,
       c.column_name,
       c.data_type,
       k.constraint_name
FROM information_schema.columns AS c 
LEFT JOIN information_schema.key_column_usage AS k 
    USING(table_name, column_name, table_schema)
WHERE c.table_schema = 'fantasy' and c.table_name = 'events'
ORDER BY c.table_name;

table_schema|	table_name|	column_name|	data_type|	constraint_name
:-------: | :-------: | :-------: | :-------: | -------:
fantasy|	events|	transaction_id|	character varying|	events_pkey
fantasy|	events|	id|	character varying|	events_id_fkey
fantasy|	events|	date|	character varying|	
fantasy|	events|	time|	character varying|	
fantasy|	events|	item_code|	integer|	events_item_code_fkey
fantasy|	events|	amount|	real|	
fantasy|	events|	seller_id|	character varying|	

Таблица `events` содержит семь полей, и большинство из них хранит текстовые данные. При этом поле `transaction_id` с идентификатором транзакции — это первичный ключ таблицы, а два поля `id` и `item_code` — внешние ключи, связывающие данные с таблицами `users` и `items`.

В полях с датой `date` и временем `time` хранятся значения типа `character varying`, а это не совсем корректно. Эту особенность нужно учитывать при работе с датой и временем.

### 1.7. Вывод первых строк таблицы events

Выведем первые пять строк таблицы `events`. При этом в выдачу добавим поле `row_count` с подсчётом общего количества строк в таблице.

In [ ]:
SELECT *, count(*) over() as row_count
FROM fantasy.events
LIMIT 5

transaction_id|id|date|time|item_code|amount|seller_id|row_count
:------- | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | -------:
2129235853|37-5938126|2021-01-03|16:31:49|6010|21.41|220381|1307678
2129237617|37-5938126|2021-01-03|16:49:00|6010|64.98|54680|1307678
2129239381|37-5938126|2021-01-03|21:05:29|6010|50.68|888909|1307678
2129241145|37-5938126|2021-01-03|22:03:02|6010|46.49|888902|1307678
2129242909|37-5938126|2021-01-03|22:04:26|6010|18.72|888905|1307678

Игроки совершили больше миллиона внутриигровых покупок — есть что анализировать. Id продавца отличается по структуре от id игрока. Видимо, для продажи эпических предметов игрок должен зарегистрироваться как продавец.

### 1.8. Проверка пропусков в таблице events

Проверим данные в таблице `events` на возможные ошибки и начнем с пропусков. Найдем количество строк с пропусками в полях, которые будут использоваться при анализе: `date`, `time`, `amount`, `seller_id`.

In [ ]:
SELECT count(*)
FROM fantasy.events
WHERE date IS NULL OR time IS NULL OR amount IS NULL OR seller_id IS NULL

В 508186 строках из 1307678 встречаются пропуски хотя бы в одном из полей. Теперь можно проверить, что это за поля.

### 1.9. Изучение пропусков в таблице events

Выведем количество данных в каждом из полей `date`, `time`, `amount`, `seller_id`. Итоговая таблица должна содержать такие поля:

-    data_count — количество строк с данными в поле date;
-    time_count — количество строк с данными в поле time;
-    amount_count — количество строк с данными в поле amount;
-    seller_id_count — количество строк с данными в поле seller_id.

In [ ]:
SELECT 
    SUM(CASE WHEN date IS NULL THEN 0 ELSE 1 END) AS data_count,
    SUM(CASE WHEN time IS NULL THEN 0 ELSE 1 END) AS data_time,
    SUM(CASE WHEN amount IS NULL THEN 0 ELSE 1 END) AS data_amount,
    SUM(CASE WHEN seller_id IS NULL THEN 0 ELSE 1 END) AS data_seller_id
FROM fantasy.events
WHERE date IS NULL
  OR time IS NULL
  OR amount IS NULL
  OR seller_id IS NULL

data_count|	data_time|	data_amount|	data_seller_id
:-------: | :-------: | :-------: | :-------: |
508186|	508186|	508186|	0

Все 508186 пропусков содержатся только в поле `seller_id`, то есть в данных нет информации о продавце. Видимо, в таком случае покупка совершалась в игровом магазине, а не у других продавцов.

### 1.10. Изучение таблиц с дополнительной информацией

Таблицы `classes`, `country`, `items`, `skills` и `race`содержат расшифровки идентификаторов:

-    класса персонажа — таблица `classes`;
-    легендарного умения — таблица `skills`;
-    расы персонажа — таблица `race`;
-    эпического предмета — таблица `items`;
-    страны регистрации игрока — таблица `country`.

Эти таблицы расшифровывают идентификаторы и коды, которые встречаются в двух других таблицах — users и events. Именно эти данные позволят выделить категории игроков, например, по стране регистрации или расе персонажа, а также проанализировать активность игроков при покупке эпических предметов. Перед анализом данных важно проверить объём значений в каждой категории.

- Данные в таблице `country`

Изучим данные таблицы `country` и проверим, игроки из каких стран зарегистрированы в игре.

In [ ]:
SELECT DISTINCT LOCATION
FROM fantasy.country

|location
|:-----:
|China
|Japan
|Russia
|South Korea
|United States

- Данные таблицы `classes`  

Изучим данные таблицы classes и узнаем, сколько уникальных классов существует в игре.

In [ ]:
SELECT DISTINCT class
FROM fantasy.classes
ORDER BY class

Всего игрокам доступно 13 классов персонажа. Это Archer, Barbarian, Bard, Druid, Engineer, Healer, Knight, Mage, Monk, Necromancer, Paladin, Rogue, Shaman.

- Данные таблицы `race`

Изучим данные таблицы `race` и узнаем, какое уникальное количество рас доступно для игрока.

In [ ]:
SELECT DISTINCT race
FROM fantasy.race
ORDER BY race

Доступных рас не так много — всего семь уникальных значений: Angel, Demon, Elf, Hobbit, Human, Northman и Orc.

- Данные таблицы `skills`

Изучим данные таблицы `skills` и узнаем, какое уникальное количество легендарных умений доступно для выбора.

In [ ]:
SELECT DISTINCT legendary_skill
FROM fantasy.skills
ORDER BY legendary_skill

Умений достаточно много — 181. Вероятно, выбор умения зависит от расы или класса игрока.

- Данные таблицы `items`

Изучим данные таблицы `items` и узнаем, каким уникальным количеством эпических предметов могут воспользоваться игроки.

In [ ]:
SELECT DISTINCT game_items
FROM fantasy.items
ORDER BY game_items

Всего 184 предмета.

Уникальные значения по разным признакам игровых персонажей и категориям пользователей можно использовать для дальнейшего анализа данных и сегментации игроков, чтобы найти среди них наиболее активных и платёжеспособных.

## 2. Исследовательский анализ данных

Команде игры «Секреты Темнолесья» важно знать, какая доля игроков покупает внутриигровую валюту за реальные деньги, то есть долю платящих игроков. Их также интересует, как игроки совершают внутриигровые покупки. 

### 2.1. Исследование доли платящих игроков

Изучим, какую долю составляют игроки, которые купили внутриигровую валюту «райские лепестки» за реальные деньги, от общего количества зарегистрированных игроков. Выгрузим несколько полей:
- общее количество игроков, зарегистрированных в игре;
- количество платящих игроков;
- доля платящих игроков от общего количества пользователей, зарегистрированных в игре

In [ ]:
SELECT count(id) AS total_count,
    sum(payer) AS paying_count,
    avg(payer)::numeric(6,4) AS part_paying
FROM fantasy.users

total_count|paying_count|part_paying|
:--------:|:--------:|:--------:|
      22214|        3929|     0.1769|

Изучим, зависит ли доля платящих игроков от выбранной расы персонажа. Выведем следующие значения для каждой расы персонажа:
- раса персонажа;
- количество платящих игроков этой расы;
- общее количество зарегистрированных игроков этой расы;
- доля платящих игроков среди всех зарегистрированных игроков этой расы.

In [ ]:
SELECT race, COUNT(id) AS race_count,
    SUM(payer) AS paying_count,
    (SUM(payer)::float / COUNT(id))::numeric(6,4) AS part_paying
FROM fantasy.users AS u
JOIN fantasy.race AS r ON u.race_id = r.race_id
GROUP BY race
ORDER BY part_paying;

race    |race_count|paying_count|part_paying|
:-------|:--------:|:--------:|:--------:|
Elf     |      2501|         427|     0.1707|
Angel   |      1327|         229|     0.1726|
Orc     |      3619|         636|     0.1757|
Northman|      3562|         626|     0.1757|
Human   |      6328|        1114|     0.1760|
Hobbit  |      3648|         659|     0.1806|
Demon   |      1229|         238|     0.1937|

Доля платящих пользователей по всем данным составляет 0,1769. Доля платящих пользователей не сильно зависит от расы, но можно выделить расу с наибольшей долей платящих пользователей — 0.19365 — Demon и с наименьшей долей — 0,17073 — Elf.  
При этом раса Demon самая малочисленная — общее количество зарегистрированных игроков этой расы составляет 1229. Самая популярная раса среди игроков — Human, ее выбрали 6328 пользователя, доля платящих пользователей этой расы близка к средней доле всех платящих игроков — 0.1760. 

### 2.2. Исследование внутриигровых покупок

Получим основные статистические показатели по полю покупки `amount`, включая:

-    общее количество покупок;
-    суммарную стоимость всех покупок;
-    минимальную и максимальную стоимость покупки;
-    среднее значение, медиану и стандартное отклонение стоимости покупки.

In [ ]:
SELECT COUNT(transaction_id) AS total_count,
    SUM(amount) AS total_sum,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount)::numeric(10,2) AS avg_amount,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY amount) AS median_amount,
    STDDEV(amount)::numeric(10,2) AS stddev_amount
FROM fantasy.events
WHERE amount <> 0

total_count|total_sum|min_amount|max_amount|avg_amount|median_amount|stddev_amount|
:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|:---------:|
    1306771|686615040|      0.01|  486615.1|    526.06|        74.86|      2518.18|

Проверим, встречаются ли покупки с нулевой стоимостью. Если да, найдем их абсолютное количество и долю от общего числа покупок. Покупки с нулевой стоимостью не помогают зарабатывать внутриигровую валюту «райские лепестки», и их следует исключить при решении следующих задач.

In [ ]:
SELECT COUNT(CASE WHEN amount = 0 THEN transaction_id ELSE NULL END) AS zero_amount,
    (COUNT(CASE WHEN amount = 0 THEN transaction_id ELSE NULL END)::float / COUNT(transaction_id))::numeric(10,5) AS part_zero
FROM fantasy.events

zero_amount|part_zero|
:---------:|:-------:|
        907|  0.00069|

Среди покупок встречаются покупки с нулевой стоимостью. Количество таких покупок — 907, что составляет 0.069% от общего числа покупок и практически  не влияет на основные статистические показатели стоимости покупок.

Проверим, какие предметы покупали с нулевой стоимостью, а также выведем количество покупок таких предметов с группировкой по пользователям.

In [ ]:
SELECT tech_nickname, race, e.item_code, game_items, count(transaction_id) AS item_count
FROM fantasy.events AS e
JOIN fantasy.items AS i ON e.item_code = i.item_code
JOIN fantasy.users AS u ON e.id = u.id
JOIN fantasy.race AS r ON u.race_id = r.race_id
WHERE amount = 0
GROUP BY tech_nickname, race, e.item_code, game_items
ORDER BY item_count DESC

tech_nickname         |race    |item_code|game_items     |item_count|
:--------:|:--------:|:--------:|:--------:|:--------:|
MajesticGuardian6128  |Elf     |     6010|Book of Legends|       810|
HeroicAvenger1612     |Angel   |     6010|Book of Legends|         6|
HeroicWarden989       |Hobbit  |     6010|Book of Legends|         6|
...  |...  |     ...|...|         ...|
GallantSwordsman2749  |Hobbit  |     6010|Book of Legends|         1|

Изучим популярность эпических предметов. Для каждого предмета посчитаем:

-    Общее количество внутриигровых продаж в абсолютном и относительном значениях. Относительное значение должно быть долей продажи каждого предмета от всех продаж.
-    Долю игроков, которые хотя бы раз покупали этот предмет, от общего числа внутриигровых покупателей.

In [ ]:
SELECT game_items AS item, 
    COUNT(transaction_id) AS count_sales,
    COUNT(transaction_id)::float / (SELECT COUNT(transaction_id) FROM fantasy.events WHERE amount <> 0) AS part_sales,
    COUNT(DISTINCT id)::float / (SELECT COUNT(DISTINCT id) FROM fantasy.events WHERE amount <> 0) AS part_users
FROM fantasy.events AS e
JOIN fantasy.items AS i ON e.item_code = i.item_code
WHERE amount <> 0
GROUP BY item
ORDER BY part_users DESC

item                     |count_sales|part_sales           |part_users           |
:-----------:|:-----------:|:-----------:|:-----------:|
Book of Legends          |    1004516|   0.7687008664869361|   0.8841357308584686|
Bag of Holding           |     271875|   0.2080509898061711|   0.8677494199535963|
Necklace of Wisdom       |      13828| 0.010581808136238102|   0.1179669373549884|
Gems of Insight          |       3833|0.0029331841615707725|  0.06714037122969838|
Treasure Map             |       3183|0.0024357748985859035| 0.059382250580046404|
...             |       ...|...| ...|

Среди эпических предметов можно выделить тройку наиболее популярных: Book of Legends, Bag of Holding и Necklace of Wisdom, на них приходится 76.87%, 20.81% и 1.06% от всех продаж соответственно.  
Доля игроков, которые хотя бы раз покупали Book of Legends, составляет 88.41%, Bag of Holding — 86.77%, Necklace of Wisdom — 11.80%.

## 3. Решение ad hoc задачи

Необходимо изучить активность игроков при покупке эпических предметов в разрезе разных рас персонажей. Есть гипотеза, что игра за некоторые расы сложнее и требует большего количества покупок эпических предметов, которые помогают в прохождении.  
Посчитаем такие показатели для каждой игровой расы:

-    общее количество зарегистрированных игроков;
-    количество игроков, которые совершают внутриигровые покупки, и их доля от общего количества зарегистрированных игроков;
-    доля платящих игроков среди игроков, которые совершили внутриигровые покупки;
-    среднее количество покупок на одного игрока, совершившего внутриигровые покупки;
-    средняя стоимость одной покупки на одного игрока, совершившего внутриигровые покупки;
-   средняя суммарная стоимость всех покупок на одного игрока, совершившего внутриигровые покупки.

In [ ]:
WITH races AS (
    SELECT race_id, count(id) AS total_count
    FROM fantasy.users AS u
    GROUP BY race_id
),
made_sale AS (
    SELECT race_id,
        COUNT(DISTINCT e.id) AS count_made_sale,
        COUNT(transaction_id) AS count_sales,
        SUM(amount) AS sum_sales,
        AVG(amount) AS avg_sales,
        COUNT(DISTINCT e.id) FILTER(WHERE payer = 1) paying_count
    FROM fantasy.events AS e
    JOIN fantasy.users AS u ON e.id = u.id
    WHERE amount <> 0
    GROUP BY race_id
)
SELECT race, total_count,
    count_made_sale AS count_made_sale,
    (count_made_sale::float / total_count)::numeric(6,4) AS part_made_sale, 
    paying_count::float / count_made_sale AS part_paying,
    count_sales::float / count_made_sale AS avg_count_sales,
    avg_sales AS avg_sum_per_user,
    sum_sales::float / count_made_sale AS avg_sum_sales
FROM races
JOIN fantasy.race AS r ON races.race_id = r.race_id
LEFT JOIN made_sale ON races.race_id = made_sale.race_id

race    |total_count|count_made_sale|part_made_sale|part_paying       |avg_count_sales   |avg_sum_per_user  |avg_sum_sales     |
:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|
Elf     |       2501|           1543|        0.6170|0.16267012313674659| 78.79066753078419| 682.3347716039249| 53760.63771872975|
Northman|       3562|           2229|        0.6258|0.18214445939883356| 82.10183938986093| 761.5012180548529|62522.027815163754|
Angel   |       1327|            820|        0.6179|0.16707317073170733| 106.8048780487805|455.67816581929185|48664.917073170735|
Orc     |       3619|           2276|        0.6289|0.17398945518453426| 81.73813708260106| 510.9002556814825| 41761.62038664323|
Hobbit  |       3648|           2266|        0.6212|  0.176963812886143|  86.1288614298323|  552.903146444979|47621.948808473084|
Human   |       6328|           3921|        0.6196|0.18005610813567968|121.40219331803111| 403.1307966460304| 48931.70925784238|
Demon   |       1229|            737|        0.5997| 0.1994572591587517| 77.86974219810041| 529.0550736998873| 41194.69199457259|

Доля игроков, которые совершают внутриигровые покупки, слабо зависит от расы персонажа и колеблется в пределах от 59.97% для расы Demon до 62.89% — для Orc.  
Доля игроков, покупающих валюту за реальные деньги, также слабо зависит от расы, максимальное значение составляет 19.95% для Demon , минимальное — 16.27% для Elf.  
Игроки расы Human чаще остальных покупают эпические предметы, в среднем 121.40 покупок на 1 игрока, наименее активная раса — Demon, в среднем 77.87 покупок на 1 игрока.  
Средняя стоимость 1 покупки на 1 игрока у расы Human составляет всего 403.13, что является наименьшей стоимостью среди всех рас.
Самые дорогие покупки совершают Northman и Elf, средняя стоимость 1 покупки на 1 игрока у них составляет 761.50 и 682.33 соответственно.  
Средняя суммарная стоимость всех покупок на одного игрока для этих 2 рас также самая высокая: 62518.17 для Northman и 53761.73 для Elf.

## 4. Общие выводы и рекомендации

Активность игроков, совершающих внутриигровые покупки, равно как и доля игроков, покупающих валюту за реальные деньги, не зависит от расы персонажа, что может свидетельствовать о сбалансированности всех рас персонажей.  

Наибольшее количество покупок эпических предметов на 1 игрока приходится на расы Human и Angel (121.40 и 106.80), при этом эти же расы покупают предметы с наименьшей стоимостью 1 покупки на 1 игрока (403.13 и 455.68). Можно рассмотреть вариант увеличения стоимости предметов для этих рас с целью стимулирования покупки игровой валюты за реальные деньги. 

Особое внимание стоит обратить на расу Demon: среди игроков этой расы самая высокая доля покупающих валюту за реальные деньги — 19.95%, при этом самая низкая средняя суммарная стоимость всех покупок на одного игрока — 41194.80. Также данная раса является самой малочисленной — 1229 игроков. Можно рекомендовать проведение рекламной акции для стимулирования выбора игроками этой расы.  

Следует обратить внимание на наличие покупок с нулевой стоимостью. Их доля мала и не влияет на основные статистические показатели, однако в ходе анализа было выявлено, что все покупки с нулевой стоимостью относятся к единственному эпическому предмету — Book of Legends. Также можно отметить, что 810 таких покупок были совершены одним игроком расы Elf с ником MajesticGuardian6128. Данные факты в совокупности могут свидетельствовать о наличии каких-то проблем с игровой механикой этого эпического предмета. Можно рекомендовать проведение детального разбора логов игры данного игрока.